# 🏓 Pickleball Ball Detection - Training (A100 Optimized)

**Mô hình:** YOLOv8m (Medium)  
**Dataset:** annotations_pickleball (v8) từ Roboflow — 18,408 ảnh  
**GPU:** A100 80GB — batch=64, workers=8  
**Mục tiêu:** Phát hiện bóng pickleball trong video

## 1. Setup & Cài đặt

In [ ]:
# Kiểm tra GPU
!nvidia-smi

In [ ]:
# Cài đặt thư viện
!pip install ultralytics roboflow supervision -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/pickleball_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

## 2. Tải Dataset từ Roboflow

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "VAwW4zaxVs978t7iLszZ"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("pickleballbadminton").project("annotations_pickleball")
version = project.version(8)
dataset = version.download("yolov8", location="/content/ball_dataset_raw")

print(f"\nDataset downloaded to: {dataset.location}")

## 3. Chia lại Dataset (70/20/10)

In [ ]:
import os
import shutil
import random

random.seed(42)

RAW_DIR = '/content/ball_dataset_raw'
SPLIT_DIR = '/content/ball_dataset'

all_images = []
all_labels = []

for split in ['train', 'valid', 'test', '']:
    img_dir = os.path.join(RAW_DIR, split, 'images') if split else os.path.join(RAW_DIR, 'images')
    lbl_dir = os.path.join(RAW_DIR, split, 'labels') if split else os.path.join(RAW_DIR, 'labels')
    
    if not os.path.exists(img_dir):
        continue
    
    for img_name in os.listdir(img_dir):
        if not img_name.endswith(('.jpg', '.jpeg', '.png')):
            continue
        
        lbl_name = os.path.splitext(img_name)[0] + '.txt'
        lbl_path = os.path.join(lbl_dir, lbl_name)
        
        if os.path.exists(lbl_path):
            all_images.append(os.path.join(img_dir, img_name))
            all_labels.append(lbl_path)

print(f'Total images found: {len(all_images)}')

combined = list(zip(all_images, all_labels))
random.shuffle(combined)

n = len(combined)
n_train = int(n * 0.7)
n_val = int(n * 0.2)

splits = {
    'train': combined[:n_train],
    'valid': combined[n_train:n_train + n_val],
    'test': combined[n_train + n_val:],
}

for split_name, items in splits.items():
    img_dst = os.path.join(SPLIT_DIR, split_name, 'images')
    lbl_dst = os.path.join(SPLIT_DIR, split_name, 'labels')
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)
    
    for img_src, lbl_src in items:
        shutil.copy2(img_src, img_dst)
        shutil.copy2(lbl_src, lbl_dst)
    
    print(f'{split_name}: {len(items)} images')

print(f'\nSplit dataset saved to: {SPLIT_DIR}')

## 4. Visualize Dữ liệu

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

def visualize_ball_detection(img_path, lbl_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    with open(lbl_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.circle(img, (int(cx * w), int(cy * h)), 3, (0, 255, 0), -1)
    return img

img_dir = os.path.join(SPLIT_DIR, 'train', 'images')
lbl_dir = os.path.join(SPLIT_DIR, 'train', 'labels')
images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
samples = random.sample(images, min(8, len(images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, img_name in enumerate(samples):
    ax = axes[idx // 4][idx % 4]
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    img_path = os.path.join(img_dir, img_name)
    lbl_path = os.path.join(lbl_dir, lbl_name)
    if os.path.exists(lbl_path):
        vis = visualize_ball_detection(img_path, lbl_path)
        ax.imshow(vis)
    ax.set_title(img_name[:20], fontsize=7)
    ax.axis('off')
plt.suptitle('Ball Detection - Sample Annotations', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Cấu hình data.yaml

In [ ]:
import yaml

data_config = {
    'path': '/content/ball_dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['ball'],
}

yaml_path = '/content/ball_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'Created: {yaml_path}')
with open(yaml_path, 'r') as f:
    print(f.read())

## 6. Huấn luyện YOLOv8m (A100 Optimized)

**Cấu hình tối ưu cho A100 80GB:**
- `batch=64` — tận dụng tối đa 80GB VRAM
- `workers=8` — load dữ liệu nhanh hơn
- `imgsz=1280` — ảnh lớn để phát hiện bóng nhỏ

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/content/ball_data.yaml',
    epochs=100,
    imgsz=1280,
    batch=64,               # A100 80GB — tận dụng tối đa VRAM
    device=0,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    mosaic=1.0,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    workers=8,              # Tăng data loading workers
    project='/content/runs/ball',
    name='train_v1',
    exist_ok=True,
)

## 7. Đánh giá Mô hình

In [ ]:
metrics = model.val(
    data='/content/ball_data.yaml',
    split='test',
    imgsz=1280,
)

print("\n" + "="*50)
print("  KẾT QUẢ ĐÁNH GIÁ")
print("="*50)
print(f"  Precision:      {metrics.box.mp:.4f}")
print(f"  Recall:         {metrics.box.mr:.4f}")
print(f"  mAP@50:         {metrics.box.map50:.4f}")
print(f"  mAP@50-95:      {metrics.box.map:.4f}")
print("="*50)

In [ ]:
from IPython.display import Image, display

results_dir = '/content/runs/ball/train_v1'

if os.path.exists(f'{results_dir}/results.png'):
    display(Image(filename=f'{results_dir}/results.png', width=900))

if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

test_images = glob.glob(f'{SPLIT_DIR}/test/images/*')[:8]
best_model = YOLO(f'{results_dir}/weights/best.pt')

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, img_path in enumerate(test_images):
    ax = axes[idx // 4][idx % 4]
    result = best_model(img_path, verbose=False, imgsz=1280)[0]
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(os.path.basename(img_path)[:20], fontsize=7)
    ax.axis('off')
plt.suptitle('Ball Detection - Test Results', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Lưu Mô hình về Google Drive

In [ ]:
import shutil

src_weights = f'{results_dir}/weights/best.pt'
dst_weights = os.path.join(SAVE_DIR, 'ball_tracker_best.pt')
shutil.copy2(src_weights, dst_weights)
print(f'✅ Best weights saved to: {dst_weights}')

src_last = f'{results_dir}/weights/last.pt'
dst_last = os.path.join(SAVE_DIR, 'ball_tracker_last.pt')
shutil.copy2(src_last, dst_last)
print(f'✅ Last weights saved to: {dst_last}')

results_dst = os.path.join(SAVE_DIR, 'ball_training_results')
if os.path.exists(results_dst):
    shutil.rmtree(results_dst)
shutil.copytree(results_dir, results_dst)
print(f'✅ Training results saved to: {results_dst}')

---
## ✅ Hoàn tất!

Copy `ball_tracker_best.pt` từ Google Drive vào `models/` rồi chạy:
```bash
python main.py --input video.mp4 \
               --court-model models/court_keypoint_best.pt \
               --ball-model models/ball_tracker_best.pt
```